# Reservoir Sampling

## Learning Objectives

1. **State** the problem: draw a uniform random sample of size $k$ from a stream of unknown length $n$
2. **Prove** that Algorithm R gives each element probability exactly $k/n$ of being in the final reservoir
3. **Extend** to weighted reservoir sampling (Efraimidis-Spirakis)
4. **Implement** both variants and verify uniformity empirically


## Problem Statement

### Sampling from an Unknown-Length Stream

We need to maintain a sample of $k$ elements from a stream where:
- The total length $n$ is **not known in advance**
- We can only read each element **once** (no rewinding)
- We must use only $O(k)$ memory

At the end, each element in the stream must have been chosen with probability exactly $k/n$.

### Applications

- Distributed systems: sample a random subset of log entries without counting logs first
- Database query processing: approximate query answers over large tables
- A/B testing: randomly assign users to treatment with probability $k/n$ without pre-knowing $n$

### Why Not Just Pick $k$ Random Indices?

We would need to know $n$ first. If we stream through the data, we must make a keep/discard decision at each element **before** knowing how many elements remain.


In [ ]:
from IPython.display import SVG, display
svg = '''
<svg xmlns="http://www.w3.org/2000/svg" width="820" height="360" font-family="monospace" font-size="12">
  <rect width="820" height="360" fill="#fafafa" rx="8"/>
  <text x="410" y="22" text-anchor="middle" fill="#333" font-size="13" font-weight="bold">Reservoir Sampling (k=3 — keep 3 elements from stream)</text>

  <!-- Phase labels -->
  <text x="410" y="48" text-anchor="middle" fill="#555" font-size="11">At each step i: keep item i with probability k/i; if kept, replace a random reservoir item</text>

  <!-- Stream and reservoir steps -->
  <!-- i=1 -->
  <rect x="20" y="60" width="70" height="30" rx="3" fill="#4e79a7"/><text x="55" y="80" text-anchor="middle" fill="white" font-size="11">item₁</text>
  <text x="110" y="78" fill="#555" font-size="10">i=1: P=3/1&gt;1 → keep (fill)</text>
  <rect x="240" y="60" width="50" height="30" rx="3" fill="#4e79a7"/><text x="265" y="80" text-anchor="middle" fill="white" font-size="11">R[0]</text>
  <rect x="295" y="60" width="50" height="30" rx="3" fill="#f5f5f5" stroke="#ccc"/><text x="320" y="80" text-anchor="middle" fill="#ccc" font-size="11">—</text>
  <rect x="350" y="60" width="50" height="30" rx="3" fill="#f5f5f5" stroke="#ccc"/><text x="375" y="80" text-anchor="middle" fill="#ccc" font-size="11">—</text>
  <text x="420" y="78" fill="#aaa" font-size="10">reservoir=[item₁, _, _]</text>

  <!-- i=2 -->
  <rect x="20" y="100" width="70" height="30" rx="3" fill="#4e79a7"/><text x="55" y="120" text-anchor="middle" fill="white" font-size="11">item₂</text>
  <text x="110" y="118" fill="#555" font-size="10">i=2: P=3/2&gt;1 → keep (fill)</text>
  <rect x="240" y="100" width="50" height="30" rx="3" fill="#4e79a7"/><text x="265" y="120" text-anchor="middle" fill="white" font-size="11">R[0]</text>
  <rect x="295" y="100" width="50" height="30" rx="3" fill="#4e79a7"/><text x="320" y="120" text-anchor="middle" fill="white" font-size="11">R[1]</text>
  <rect x="350" y="100" width="50" height="30" rx="3" fill="#f5f5f5" stroke="#ccc"/><text x="375" y="120" text-anchor="middle" fill="#ccc" font-size="11">—</text>
  <text x="420" y="118" fill="#aaa" font-size="10">reservoir=[item₁, item₂, _]</text>

  <!-- i=3 -->
  <rect x="20" y="140" width="70" height="30" rx="3" fill="#4e79a7"/><text x="55" y="160" text-anchor="middle" fill="white" font-size="11">item₃</text>
  <text x="110" y="158" fill="#555" font-size="10">i=3: P=3/3=1 → keep (fill)</text>
  <rect x="240" y="140" width="50" height="30" rx="3" fill="#4e79a7"/><text x="265" y="160" text-anchor="middle" fill="white" font-size="11">R[0]</text>
  <rect x="295" y="140" width="50" height="30" rx="3" fill="#4e79a7"/><text x="320" y="160" text-anchor="middle" fill="white" font-size="11">R[1]</text>
  <rect x="350" y="140" width="50" height="30" rx="3" fill="#4e79a7"/><text x="375" y="160" text-anchor="middle" fill="white" font-size="11">R[2]</text>
  <text x="420" y="158" fill="#aaa" font-size="10">reservoir=[item₁, item₂, item₃] (full)</text>

  <!-- i=4 -->
  <rect x="20" y="180" width="70" height="30" rx="3" fill="#f28e2b"/><text x="55" y="200" text-anchor="middle" fill="white" font-size="11">item₄</text>
  <text x="110" y="198" fill="#555" font-size="10">i=4: P=3/4=0.75 → keep, replace R[rand(0..2)]</text>
  <rect x="240" y="180" width="50" height="30" rx="3" fill="#4e79a7"/><text x="265" y="200" text-anchor="middle" fill="white" font-size="11">R[0]</text>
  <rect x="295" y="180" width="50" height="30" rx="3" fill="#f28e2b"/><text x="320" y="200" text-anchor="middle" fill="white" font-size="11">R[1]</text>
  <rect x="350" y="180" width="50" height="30" rx="3" fill="#4e79a7"/><text x="375" y="200" text-anchor="middle" fill="white" font-size="11">R[2]</text>
  <text x="420" y="198" fill="#aaa" font-size="10">item₄ replaced item₂ at slot 1</text>

  <!-- i=5 skip -->
  <rect x="20" y="220" width="70" height="30" rx="3" fill="#ccc"/><text x="55" y="240" text-anchor="middle" fill="white" font-size="11">item₅</text>
  <text x="110" y="238" fill="#555" font-size="10">i=5: P=3/5=0.60 → skip (no change)</text>
  <rect x="240" y="220" width="50" height="30" rx="3" fill="#4e79a7"/><text x="265" y="240" text-anchor="middle" fill="white" font-size="11">R[0]</text>
  <rect x="295" y="220" width="50" height="30" rx="3" fill="#f28e2b"/><text x="320" y="240" text-anchor="middle" fill="white" font-size="11">R[1]</text>
  <rect x="350" y="220" width="50" height="30" rx="3" fill="#4e79a7"/><text x="375" y="240" text-anchor="middle" fill="white" font-size="11">R[2]</text>
  <text x="420" y="238" fill="#aaa" font-size="10">item₅ discarded</text>

  <!-- Proof box -->
  <rect x="20" y="268" width="780" height="80" rx="5" fill="#e8f8e8" stroke="#59a14f" stroke-width="1.5"/>
  <text x="410" y="288" text-anchor="middle" fill="#59a14f" font-size="12" font-weight="bold">Correctness: Each element i is in final reservoir with probability k/n</text>
  <text x="410" y="308" text-anchor="middle" fill="#333" font-size="11">P[item i survives] = P[item i kept at step i] × P[not replaced after]</text>
  <text x="410" y="328" text-anchor="middle" fill="#333" font-size="11">= (k/i) × (i/(i+1)) × ((i+1)/(i+2)) × ... × ((n-1)/n)  =  k/n   ✓</text>
  <text x="410" y="344" text-anchor="middle" fill="#555" font-size="10">(telescoping product)</text>
</svg>
'''
display(SVG(svg))


## Derivation — Algorithm R (Vitter 1985)

### Algorithm

1. **Fill phase:** Put elements $1, \ldots, k$ directly into the reservoir
2. **Stream phase:** For element $i > k$: with probability $k/i$, pick a random slot $j \in \{1, \ldots, k\}$ and replace reservoir$[j]$ with element $i$; otherwise discard

### Proof of Correctness

Claim: after processing $n$ elements, each element has probability exactly $k/n$ of being in the reservoir.

**For element $i \leq k$** (fill phase):
$$P[i \text{ survives}] = P[i \text{ not replaced at step } k+1] \times \cdots \times P[i \text{ not replaced at step } n]$$

At step $t > k$: element $i$ is replaced with probability $\frac{k}{t} \cdot \frac{1}{k} = \frac{1}{t}$, so it survives with probability $\frac{t-1}{t}$.

$$P[i \text{ survives}] = \frac{k}{k+1} \cdot \frac{k+1}{k+2} \cdots \frac{n-1}{n} = \frac{k}{n} \quad \checkmark$$

**For element $i > k$** (stream phase):
$$P[i \text{ selected}] \times P[i \text{ not replaced later}] = \frac{k}{i} \times \frac{i}{i+1} \times \cdots \times \frac{n-1}{n} = \frac{k}{n} \quad \checkmark$$

### Complexity

- **Time:** $O(n)$ — one pass through stream
- **Space:** $O(k)$ — just the reservoir
- **Decisions per element:** $O(1)$ — one random number + conditional replacement


## Algorithm Steps

1. **Initialise** reservoir $R = [\text{element}_1, \ldots, \text{element}_k]$
2. **For each element** $x_i$ with $i > k$:
   - Generate $j = \text{randint}(0, i)$ uniformly
   - If $j < k$: set $R[j] = x_i$ (replace reservoir slot $j$)
3. **Return** $R$ — a uniform random sample of size $k$


In [ ]:
import random


def reservoir_sample(stream, k):
    """
    Reservoir sampling: draw a uniform random sample of size k
    from a stream of unknown length.

    Inputs
    ------
    stream : iterable — the data stream (read once, left to right)
    k      : int — sample size

    Output
    ------
    reservoir : list of k elements chosen uniformly at random
    """
    reservoir = []

    for i, item in enumerate(stream):
        if i < k:
            # Fill reservoir with first k elements
            reservoir.append(item)
        else:
            # Replace reservoir[j] with probability k/(i+1)
            j = random.randint(0, i)
            if j < k:
                reservoir[j] = item

    return reservoir


def reservoir_sample_weighted(stream, k):
    """
    Weighted reservoir sampling (Efraimidis-Spirakis A-ES algorithm).
    Each item has a weight; items with higher weight are more likely to be sampled.

    stream : iterable of (item, weight) pairs
    k      : sample size
    """
    import heapq

    # Priority queue: (key, item) — keep k largest keys
    heap = []

    for item, weight in stream:
        # Key = u^(1/weight) where u ~ Uniform(0,1)
        u = random.random()
        key = u ** (1.0 / weight) if weight > 0 else 0.0

        if len(heap) < k:
            heapq.heappush(heap, (key, item))
        elif key > heap[0][0]:
            heapq.heapreplace(heap, (key, item))

    return [item for _, item in heap]


# ── Demo ──────────────────────────────────────────────────────────────────────
import random, collections
random.seed(42)

n = 100_000
k = 1000
stream = range(n)  # elements 0..n-1

# Run many trials and check uniformity
trials = 50
counts = collections.Counter()
for _ in range(trials):
    sample = reservoir_sample(stream, k)
    for item in sample:
        counts[item] += 1

# Each element should appear in roughly (k/n)*trials = 0.5 trials on average
expected = k / n * trials
freqs = sorted(counts.values())
print(f"Reservoir sampling (k={k}, n={n}, {trials} trials)")
print(f"Expected freq per element: {expected:.2f}")
print(f"Observed freq — min: {freqs[0]}, median: {freqs[len(freqs)//2]}, max: {freqs[-1]}")
print(f"% elements never sampled: {(n - len(counts)) / n * 100:.1f}%  (expected ~{(1 - k/n)**trials*100:.1f}%)")
